# ST-Reason · 01 Data preparation (week 1)

This notebook builds every dataset file the project needs and saves it to Drive:

| Step | Output on Drive |
| --- | --- |
| 1. Download annotations (NExT-QA, NExT-GQA, IntentQA) | `data/annotations/{nextqa,nextgqa,intentqa}/` |
| 2. Build clean question tables | `data/annotations/questions_*.csv` |
| 3. Sample the evaluation sets (video-disjoint, frozen once created) | `data/subsets/*.csv` |
| 4. Anchor sheets for the NExT-QA gap labels | `data/annotations/anchors/` |
| 5. VidOR annotations and coverage | `data/annotations/vidor/`, `data/subsets/vidor_coverage.json` |
| 6. Extract only the needed videos | `data/videos_nextqa_eval.zip`, `data/videos_intentqa_eval.zip` |
| 7. ST-Gap templates and clip registration | `data/stgap/` |
| 8. Data card | `data/subsets/data_card.json` |

**Runtime:** CPU is enough (Runtime → Change runtime type → CPU). Every step can be re-run safely: finished work is skipped, and the evaluation sets are never re-sampled once saved.

**Before step 6:** open the [NExT-QA video zip](https://drive.google.com/file/d/1jTcRCrVHS66ckOUfWRb-rXdzJ52XAWQH/view) in Google Drive and choose **Organize → Add shortcut**, placing it in `MyDrive/ST-Reason/data/source/`. A shortcut uses none of your Drive storage.

In [1]:
# --- 0. Bootstrap: mount Drive and load the shared helpers from 00_setup_and_utils ------------
import os, glob
ST_NOTEBOOK = '01_data_prep'
ST_QUIET = True
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')
    SETUP_NB = '/content/drive/MyDrive/ST-Reason/notebooks/00_setup_and_utils.ipynb'
elif os.path.exists('/kaggle/working'):
    SETUP_NB = (glob.glob('/kaggle/input/**/00_setup_and_utils.ipynb', recursive=True) or ['00_setup_and_utils.ipynb'])[0]
else:
    SETUP_NB = os.environ.get('ST_SETUP_NB', '00_setup_and_utils.ipynb')
%run -i $SETUP_NB

import pandas as pd
set_seed()
RUN = Run(params={'step': 'data_prep'})
ANN = P['ann']

Mounted at /content/drive
Runtime: colab | project folder: /content/drive/MyDrive/ST-Reason | local scratch: /content/st_local
config hash: f5293d4309
[09:50:39] INFO  run started: 01_data_prep-20261015-095028-f05c0b


## 1. Annotations

NExT-QA's train/val/test question files, the NExT-GQA grounding labels and the video-path map all come from the NExT-GQA repository (its `nextqa/` copy stores answers as text). IntentQA comes from its own repository. The exact commits are recorded in `SOURCES.json`.

In [2]:
def clone(name, url):
    dest = P['local_repos'] / name
    if not (dest / '.git').exists():
        subprocess.run(['git', 'clone', '--depth', '1', '-q', url, str(dest)], check=True)
    commit = subprocess.run(['git', '-C', str(dest), 'rev-parse', 'HEAD'],
                            capture_output=True, text=True).stdout.strip()
    return dest, commit

gqa_repo, gqa_commit = clone('NExT-GQA', CFG['sources']['nextgqa_repo'])
iqa_repo, iqa_commit = clone('IntentQA', CFG['sources']['intentqa_repo'])

COPY = {
    'nextqa':   [(gqa_repo / 'datasets/nextqa' / f, f) for f in ['train.csv', 'val.csv', 'test.csv']],
    'nextgqa':  [(gqa_repo / 'datasets/nextgqa' / f, f) for f in
                 ['val.csv', 'test.csv', 'gsub_val.json', 'gsub_test.json', 'map_vid_vidorID.json',
                  'frame2time_val.json', 'frame2time_test.json']],
    'intentqa': [(iqa_repo / 'datasets/IntentQA' / f, f) for f in ['train.csv', 'val.csv', 'test.csv']],
}
for folder, files in COPY.items():
    (ANN / folder).mkdir(parents=True, exist_ok=True)
    for src, name in files:
        dst = ANN / folder / name
        if not dst.exists():
            shutil.copy2(src, dst)
save_json({'NExT-GQA': {'url': CFG['sources']['nextgqa_repo'], 'commit': gqa_commit},
           'IntentQA': {'url': CFG['sources']['intentqa_repo'], 'commit': iqa_commit},
           'retrieved': dt.date.today().isoformat()}, ANN / 'SOURCES.json')
log(f'annotations ready (NExT-GQA {gqa_commit[:8]}, IntentQA {iqa_commit[:8]})')

[09:50:51] INFO  annotations ready (NExT-GQA 63772e02, IntentQA 91123039)


## 2. Clean question tables

One row per question with a stable `uid` (`<video_id>_<qid>`), the five options, the answer index, the question group (causal / temporal / descriptive), the NExT-GQA evidence segments where they exist, and a detected temporal **anchor**.

Anchor rule: a temporal question gets `anchor_type = event` when it names another event ("after the boy put down the book"), `positional` when it only names a position in the video ("at the end"), and `none` otherwise. Only `event` anchors get a NExT-QA gap label later.

Questions whose correct answer text matches two options are flagged `ambiguous` and never sampled.

In [3]:
TYPE_GROUP = {'CW': 'causal', 'CH': 'causal', 'TN': 'temporal', 'TC': 'temporal', 'TP': 'temporal',
              'DL': 'descriptive', 'DC': 'descriptive', 'DO': 'descriptive', 'DB': 'descriptive'}
OPTS = [f'a{i}' for i in range(5)]
ANCHOR_KW = re.compile(r'\b(after|before|while|when|during|until|as)\b\s+(.*)$')
POSITIONAL_TAIL = re.compile(
    r'\s*\b(at the (very )?(start|beginning|end|middle)( of the video)?|in the (beginning|middle|end)( of the video)?'
    r'|initially|first|at first|later|again|in the video)\s*$')
POSITIONAL_ONLY = re.compile(r'^(the )?(video )?(starts?|begins?|ends?|started|ended)\b|^(it|this) (starts?|ends?)')

def norm(s):
    return re.sub(r'\s+', ' ', str(s).strip().lower())

def detect_anchor(question, group):
    if group != 'temporal':
        return 'none', None, None
    m = ANCHOR_KW.search(norm(question))
    if not m:
        return ('positional' if re.search(r'\b(start|beginning|end|middle)\b', norm(question)) else 'none'), None, None
    kw, phrase = m.group(1), m.group(2)
    prev = None
    while prev != phrase:                       # strip trailing positional words, e.g. "... at the end"
        prev, phrase = phrase, POSITIONAL_TAIL.sub('', phrase).strip()
    if len(phrase.split()) < 2 or POSITIONAL_ONLY.search(phrase):
        return 'positional', kw, phrase or None
    return 'event', kw, phrase

def build_nextqa(split):
    df = pd.read_csv(ANN / 'nextqa' / f'{split}.csv', dtype={'video_id': str})
    vmap = load_json(ANN / 'nextgqa' / 'map_vid_vidorID.json')
    gsub = load_json(ANN / 'nextgqa' / f'gsub_{split}.json', default={}) if split != 'train' else {}
    rows = []
    for r in df.itertuples(index=False):
        opts = [str(getattr(r, o)) for o in OPTS]
        hits = [i for i, o in enumerate(opts) if norm(o) == norm(r.answer)]
        g = gsub.get(r.video_id, {})
        segs = g.get('location', {}).get(str(r.qid))
        group = TYPE_GROUP.get(r.type, 'other')
        a_type, a_kw, a_phrase = detect_anchor(r.question, group)
        rows.append({
            'uid': f'{r.video_id}_{r.qid}', 'dataset': 'nextqa', 'split': split,
            'video_id': r.video_id, 'qid': int(r.qid), 'type': r.type, 'group': group,
            'question': r.question, **dict(zip(OPTS, opts)),
            'answer_idx': hits[0] if len(hits) == 1 else -1, 'ambiguous': len(hits) != 1,
            'has_ground': segs is not None, 'segments': json.dumps(segs) if segs is not None else '',
            'duration_s': g.get('duration'), 'vidor_path': vmap.get(r.video_id, ''),
            'anchor_type': a_type, 'anchor_kw': a_kw or '', 'anchor_phrase': a_phrase or '',
        })
    out = pd.DataFrame(rows).drop_duplicates('uid')
    return out

def build_intentqa(split, nextqa_split_of):
    df = pd.read_csv(ANN / 'intentqa' / f'{split}.csv', dtype={'video_id': str})
    out = pd.DataFrame({
        'uid': 'iqa_' + df.video_id + '_' + df.qid.astype(str), 'dataset': 'intentqa', 'split': split,
        'video_id': df.video_id, 'qid': df.qid.astype(int), 'type': df.type,
        'group': df.type.map(TYPE_GROUP).fillna('other'), 'question': df.question,
        **{o: df[o].astype(str) for o in OPTS},
        'answer_idx': df.answer.astype(int), 'ambiguous': False,
        'nextqa_split': df.video_id.map(nextqa_split_of).fillna(''),
    })
    return out.drop_duplicates('uid')

Q = pd.concat([build_nextqa(s) for s in ['train', 'val', 'test']], ignore_index=True)
nextqa_split_of = dict(zip(Q.video_id, Q.split))
IQ = pd.concat([build_intentqa(s, nextqa_split_of) for s in ['train', 'val', 'test']], ignore_index=True)

Q.to_csv(ANN / 'questions_nextqa.csv', index=False)
IQ.to_csv(ANN / 'questions_intentqa.csv', index=False)

print('NExT-QA questions per split:', Q.split.value_counts().to_dict())
print('ambiguous answers (excluded):', int(Q.ambiguous.sum()))
print('with NExT-GQA evidence:', Q[Q.has_ground].groupby('split').size().to_dict())
print('anchor types (temporal, test):', Q[(Q.split == 'test') & (Q.group == 'temporal')].anchor_type.value_counts().to_dict())
print('IntentQA test videos by NExT-QA split:', IQ[IQ.split == 'test'].drop_duplicates('video_id').nextqa_split.value_counts().to_dict())

/tmp/ipykernel_1874/3775399726.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Q = pd.concat([build_nextqa(s) for s in ['train', 'val', 'test']], ignore_index=True)


NExT-QA questions per split: {'train': 34131, 'test': 8564, 'val': 4996}
ambiguous answers (excluded): 11
with NExT-GQA evidence: {'test': 5553, 'val': 3358}
anchor types (temporal, test): {'event': 2191, 'none': 315, 'positional': 151}
IntentQA test videos by NExT-QA split: {'train': 500, 'val': 67}


**Leakage note (IntentQA).** Every IntentQA test video is also a NExT-QA train or val video. So:

- the QLoRA model (trained on NExT-QA train) is **never** evaluated on IntentQA;
- IntentQA videos are excluded from the dev set, so tuning never sees them;
- base (untrained) models can still be evaluated on IntentQA as a generalisation check.

## 3. Evaluation sets (pre-registered)

Sampling is seeded and **video-disjoint**: no video appears in two sets. At most 4 questions are taken per video, which spreads questions across more videos. Within each video, rarer question types are taken first, so rare types such as TP are still filled.

Once `data/subsets/` holds the sets, this cell **loads them instead of re-sampling**. That freezes the questions before any result is seen. To re-sample deliberately, set `FORCE_RESAMPLE = True` and record why in `notes`.

In [4]:
FORCE_RESAMPLE = False
SUB = P['subsets']
SET_NAMES = ['dev', 'main_grid', 'descriptive', 'pilot', 'robustness', 'intentqa_eval']

def sample_video_disjoint(df, targets, cap, seed, exclude_videos=()):
    """Greedy, seeded, video-disjoint stratified sampling with a per-video cap."""
    rng = random.Random(seed)
    df = df[~df.video_id.isin(set(exclude_videos))]
    rarity = df.type.value_counts().to_dict()
    vids = sorted(df.video_id.unique()); rng.shuffle(vids)
    groups = {v: g for v, g in df.groupby('video_id')}
    need, chosen = dict(targets), []
    for v in vids:
        if all(n <= 0 for n in need.values()):
            break
        g = groups[v].sample(frac=1, random_state=rng.randint(0, 2 ** 31 - 1))
        g = g.assign(_r=g.type.map(rarity)).sort_values('_r', kind='stable')
        taken = 0
        for _, r in g.iterrows():
            if taken >= cap:
                break
            if need.get(r.type, 0) > 0:
                chosen.append(r.drop(labels='_r')); need[r.type] -= 1; taken += 1
    short = {k: n for k, n in need.items() if n > 0}
    return pd.DataFrame(chosen).reset_index(drop=True), short

def stratified_subset(df, n, seed):
    """Exactly n rows (or all rows if fewer), allocated to types in proportion to their counts."""
    n = min(n, len(df))
    counts = df.type.value_counts()
    alloc = (counts / counts.sum() * n).round().astype(int).clip(upper=counts)
    order = list(counts.index)
    i = 0
    while alloc.sum() != n:                     # fix rounding drift, largest types first
        t = order[i % len(order)]
        if alloc.sum() < n and alloc[t] < counts[t]:
            alloc[t] += 1
        elif alloc.sum() > n and alloc[t] > 0:
            alloc[t] -= 1
        i += 1
    parts = [df[df.type == t].sample(n=int(k), random_state=seed) for t, k in alloc.items() if k > 0]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

existing = {n: SUB / f'{n}.csv' for n in SET_NAMES}
if all(p.exists() for p in existing.values()) and not FORCE_RESAMPLE:
    SETS = {n: pd.read_csv(p, dtype={'video_id': str}) for n, p in existing.items()}
    log('evaluation sets already frozen: loaded from Drive (not re-sampled)')
else:
    D = CFG['data']; seed = CFG['seed']; cap = D['max_questions_per_video']
    usable = Q[~Q.ambiguous]
    iqa_videos = set(IQ[IQ.split == 'test'].video_id)

    pool_dev = usable[(usable.split == 'val') & usable.has_ground & usable.group.isin(['causal', 'temporal'])]
    dev, short_dev = sample_video_disjoint(pool_dev, D['dev_targets'], cap, seed, exclude_videos=iqa_videos)

    pool_grid = usable[(usable.split == 'test') & usable.has_ground & usable.group.isin(['causal', 'temporal'])]
    grid, short_grid = sample_video_disjoint(pool_grid, D['grid_targets'], cap, seed + 1)

    pool_desc = usable[(usable.split == 'test') & (usable.group == 'descriptive')]
    desc, short_desc = sample_video_disjoint(pool_desc, D['descriptive_targets'], D['descriptive_max_per_video'],
                                             seed + 2, exclude_videos=set(grid.video_id))

    # pilot: the first N dev videos, all their labelled causal/temporal val questions (capped)
    pilot_vids = list(dict.fromkeys(dev.video_id))[:D['pilot_videos']]
    pilot = pool_dev[pool_dev.video_id.isin(pilot_vids)].head(D['pilot_max_questions']).reset_index(drop=True)

    robust = stratified_subset(grid, D['robustness_size'], seed + 3)

    iqa_pool = IQ[(IQ.split == 'test') & ~IQ.video_id.isin(set(dev.video_id) | set(grid.video_id) | set(desc.video_id))]
    iqa_eval = stratified_subset(iqa_pool, D['intentqa_eval_size'], seed + 4)

    SETS = {'dev': dev, 'main_grid': grid, 'descriptive': desc, 'pilot': pilot,
            'robustness': robust, 'intentqa_eval': iqa_eval}
    for n, d in SETS.items():
        d.to_csv(SUB / f'{n}.csv', index=False)
    save_json({'seed': seed, 'created': dt.datetime.now().isoformat(timespec='seconds'),
               'config_hash': config_hash(), 'shortfalls': {'dev': short_dev, 'main_grid': short_grid,
               'descriptive': short_desc}, 'notes': ''}, SUB / 'sampling_record.json')
    log('evaluation sets sampled and frozen')

# --- integrity checks: these must always hold -------------------------------------------------
V = {n: set(d.video_id) for n, d in SETS.items()}
for a, b in [('dev', 'main_grid'), ('dev', 'descriptive'), ('main_grid', 'descriptive'),
             ('intentqa_eval', 'dev'), ('intentqa_eval', 'main_grid'), ('intentqa_eval', 'descriptive')]:
    assert not (V[a] & V[b]), f'video overlap between {a} and {b}'
assert V['pilot'] <= V['dev'] and V['robustness'] <= V['main_grid']
assert set(SETS['dev'].video_id).isdisjoint(set(IQ[IQ.split == 'test'].video_id))

hashes = {n: hashlib.sha1(pd.read_csv(SUB / f'{n}.csv').uid.astype(str).str.cat(sep='|').encode()).hexdigest()[:12]
          for n in SET_NAMES}
save_json(hashes, SUB / 'set_hashes.json')

summary = pd.DataFrame({n: {'questions': len(d), 'videos': d.video_id.nunique(),
                            'types': json.dumps(d.type.value_counts().sort_index().to_dict())}
                        for n, d in SETS.items()}).T
print(summary.to_string())
print('\nset hashes (quote these in the paper\'s reproducibility note):', hashes)

[09:51:04] INFO  evaluation sets sampled and frozen
              questions videos                                                   types
dev                 300     92     {"CH": 50, "CW": 110, "TC": 50, "TN": 70, "TP": 20}
main_grid           600    189  {"CH": 100, "CW": 220, "TC": 100, "TN": 140, "TP": 40}
descriptive         200    129                          {"DC": 50, "DL": 70, "DO": 80}
pilot                50     10        {"CH": 6, "CW": 16, "TC": 12, "TN": 14, "TP": 2}
robustness          200    126      {"CH": 33, "CW": 74, "TC": 33, "TN": 47, "TP": 13}
intentqa_eval       300    224                {"CH": 50, "CW": 176, "TN": 71, "TP": 3}

set hashes (quote these in the paper's reproducibility note): {'dev': '2a1a35a7a81b', 'main_grid': '2cace594d2fa', 'descriptive': 'cca82db68fce', 'pilot': '029a091cb624', 'robustness': '3ffca5ba3773', 'intentqa_eval': '999a847596f7'}


## 4. Anchor sheets for NExT-QA gap labels

Gap labels on NExT-QA need the time of the anchor event, marked by people rather than by our pipeline. This step writes two identical sheets, A and B, one per annotator. They list every main-grid temporal question with an `event` anchor. The NExT-GQA evidence segment is deliberately **not shown**, so it cannot bias where the anchor is marked.

**How to fill a sheet** (open the CSV in Google Sheets, then save back as CSV with the same name):
- Watch the video and set `anchor_start_s` and `anchor_end_s` to the time span of the anchor phrase.
- If the anchor event cannot be found, put `1` in `cannot_find`.
- Annotators fill their own sheet without looking at the other.

Sheets are never overwritten once created. The next cell computes gap labels whenever both sheets have entries.

In [5]:
ANCH = ANN / 'anchors'
ANCH.mkdir(exist_ok=True)
anchored = SETS['main_grid'][(SETS['main_grid'].group == 'temporal') & (SETS['main_grid'].anchor_type == 'event')]
sheet_cols = ['uid', 'video_id', 'question', 'anchor_phrase', 'duration_s',
              'anchor_start_s', 'anchor_end_s', 'cannot_find', 'notes']
for who in ['A', 'B']:
    path = ANCH / f'anchor_sheet_{who}.csv'
    if not path.exists():
        s = anchored.reindex(columns=sheet_cols)
        s.to_csv(path, index=False)
        log(f'created {path.name} with {len(s)} questions')
    else:
        log(f'{path.name} exists: left untouched')
print(f'{len(anchored)} anchored temporal questions in the main grid '
      f'({len(SETS["main_grid"][SETS["main_grid"].group == "temporal"])} temporal in total)')

[09:51:05] INFO  created anchor_sheet_A.csv with 246 questions
[09:51:05] INFO  created anchor_sheet_B.csv with 246 questions
246 anchored temporal questions in the main grid (280 temporal in total)


In [6]:
def interval_gap(a, b):
    """Seconds between two intervals; 0 if they overlap."""
    return max(0.0, max(a[0], b[0]) - min(a[1], b[1]))

def compute_gap_labels():
    """Combine sheets A and B. Agreed anchors (both ends within tolerance) are averaged;
    disagreements go to anchor_disagreements.csv, where filled `resolved_*` values are used."""
    tol = CFG['data']['anchor_agreement_tolerance_s']
    A = pd.read_csv(ANCH / 'anchor_sheet_A.csv', dtype={'video_id': str})
    B = pd.read_csv(ANCH / 'anchor_sheet_B.csv', dtype={'video_id': str})
    m = A.merge(B, on='uid', suffixes=('_a', '_b'))
    both = m.dropna(subset=['anchor_start_s_a', 'anchor_end_s_a', 'anchor_start_s_b', 'anchor_end_s_b'])
    if both.empty:
        print('No question has been marked by both annotators yet.')
        return None
    agree = ((both.anchor_start_s_a - both.anchor_start_s_b).abs() <= tol) & \
            ((both.anchor_end_s_a - both.anchor_end_s_b).abs() <= tol)
    res_path = ANCH / 'anchor_disagreements.csv'
    resolved = pd.read_csv(res_path) if res_path.exists() else pd.DataFrame(columns=['uid', 'resolved_start_s', 'resolved_end_s'])
    dis = both[~agree][['uid', 'anchor_start_s_a', 'anchor_end_s_a', 'anchor_start_s_b', 'anchor_end_s_b']]
    dis = dis.merge(resolved[['uid', 'resolved_start_s', 'resolved_end_s']], on='uid', how='left')
    dis.to_csv(res_path, index=False)

    grid = SETS['main_grid'].set_index('uid')
    rows = []
    for r in both.itertuples(index=False):
        ok = (abs(r.anchor_start_s_a - r.anchor_start_s_b) <= tol) and (abs(r.anchor_end_s_a - r.anchor_end_s_b) <= tol)
        if ok:
            s, e, how = (r.anchor_start_s_a + r.anchor_start_s_b) / 2, (r.anchor_end_s_a + r.anchor_end_s_b) / 2, 'agreed'
        else:
            rr = dis[dis.uid == r.uid].iloc[0]
            if pd.isna(rr.resolved_start_s) or pd.isna(rr.resolved_end_s):
                continue
            s, e, how = float(rr.resolved_start_s), float(rr.resolved_end_s), 'resolved'
        segs = json.loads(grid.loc[r.uid, 'segments'])
        gap = min(interval_gap((s, e), tuple(seg)) for seg in segs)
        rows.append({'uid': r.uid, 'anchor_start_s': s, 'anchor_end_s': e, 'gap_s': round(gap, 2), 'source': how})
    G = pd.DataFrame(rows)
    # Fixed thresholds aligned with ST-Gap (short <= 3 s, long >= 20 s). Tertiles are not used:
    # many anchors overlap their evidence (gap 0), which makes tertile cut points collapse.
    short_max, long_min = CFG['data']['gap_short_max_s'], CFG['data']['gap_long_min_s']
    if len(G):
        G['gap_bucket'] = pd.cut(G.gap_s, [-0.01, short_max, long_min - 1e-9, float('inf')],
                                 labels=['short', 'medium', 'long'])
    G.to_csv(SUB / 'nextqa_gap_labels.csv', index=False)
    if len(G):
        print('gap buckets:', G.gap_bucket.value_counts().reindex(['short', 'medium', 'long']).to_dict())
    print(f'{len(both)} marked by both | agreement {agree.mean():.0%} | {len(G)} gap labels | '
          f'{len(dis)} disagreements ({dis.resolved_start_s.notna().sum()} resolved)')
    return G

if (ANCH / 'anchor_sheet_A.csv').exists():
    compute_gap_labels()

No question has been marked by both annotators yet.


## 5. VidOR annotations and coverage

VidOR boxes and relations give ground truth for the CV metrics (experiment E1). The train and val annotation zips are downloaded from Hugging Face, and only the JSON files for our videos are kept on Drive. VidOR's test annotations are not public, so coverage of the NExT-QA test videos is measured here, as the pilot requires.

If Hugging Face cannot be reached, the step is skipped with a warning. You can then download `training-annotation.zip` and `validation-annotation.zip` from [huggingface.co/datasets/shangxd/vidor](https://huggingface.co/datasets/shangxd/vidor/tree/main) yourself, upload them to `data/source/`, and re-run this cell. They are used in preference to downloading.

In [7]:
VIDOR = ANN / 'vidor'
VIDOR.mkdir(exist_ok=True)
need = pd.concat([d[['video_id']].assign(set=n) for n, d in SETS.items()], ignore_index=True)
vmap = load_json(ANN / 'nextgqa' / 'map_vid_vidorID.json')
need['vidor_path'] = need.video_id.map(vmap)

def index_vidor_zip(zp, index):
    with zipfile.ZipFile(zp) as zf:
        for n in zf.namelist():
            if n.endswith('.json'):
                parts = Path(n).parts
                index['/'.join(parts[-2:])[:-5]] = (str(zp), n)          # key like '0000/2401075277'

vidor_index = {}
manual = sorted(P['source'].glob('*annotation*.zip'))      # optional: zips uploaded by hand
if manual:
    for zp in manual:
        index_vidor_zip(zp, vidor_index)
    log(f'VidOR annotations indexed from data/source/: {len(vidor_index)} videos')
else:
    try:
        pip_install('huggingface_hub')
        from huggingface_hub import hf_hub_download
        for name in ['training-annotation.zip', 'validation-annotation.zip']:
            zp = hf_hub_download(CFG['sources']['vidor_hf_repo'], name, repo_type='dataset',
                                 local_dir=str(P['local_tmp'] / 'vidor'))
            index_vidor_zip(zp, vidor_index)
        log(f'VidOR annotations indexed from Hugging Face: {len(vidor_index)} videos')
    except Exception as e:
        log(f'VidOR download skipped ({e.__class__.__name__}: {e}). Re-run later, or download '
            'training-annotation.zip and validation-annotation.zip from huggingface.co/datasets/shangxd/vidor '
            'into data/source/.', 'WARN')

if vidor_index:
    kept = 0
    for vid, vp in need.drop_duplicates('video_id')[['video_id', 'vidor_path']].itertuples(index=False):
        out = VIDOR / f'{vid}.json'
        if vp in vidor_index and not out.exists():
            zp, member = vidor_index[vp]
            with zipfile.ZipFile(zp) as zf:
                out.write_bytes(zf.read(member))
            kept += 1
    have = {p.stem for p in VIDOR.glob('*.json')}
    cov = {n: {'videos': len(V[n]), 'with_vidor': len(V[n] & have),
               'share': round(len(V[n] & have) / max(1, len(V[n])), 3)} for n in V}
    save_json(cov, SUB / 'vidor_coverage.json')
    print(f'{kept} new VidOR files saved. Coverage by set:')
    for n, c in cov.items():
        print(f"  {n:14s} {c['with_vidor']}/{c['videos']} videos ({c['share']:.0%})")
    if cov['main_grid']['share'] < 0.5:
        print('\nLow coverage of the main grid: E1 CV metrics will use dev-set videos (NExT-QA val) plus the '
              'ST-Gap box clips, as the plan\'s fallback allows. Record this in the paper.')

[09:51:05] INFO  pip install huggingface_hub


training-annotation.zip: reconstructing file:   0%|          |  0.00B /  151MB            

training-annotation.zip: downloading bytes:           |  0.00B            

validation-annotation.zip: reconstructing file:   0%|          |  0.00B / 17.2MB            

validation-annotation.zip: downloading bytes:           |  0.00B            

[09:51:27] INFO  VidOR annotations indexed from Hugging Face: 7835 videos
298 new VidOR files saved. Coverage by set:
  dev            92/92 videos (100%)
  main_grid      0/189 videos (0%)
  descriptive    0/129 videos (0%)
  pilot          10/10 videos (100%)
  robustness     0/126 videos (0%)
  intentqa_eval  206/224 videos (92%)

Low coverage of the main grid: E1 CV metrics will use dev-set videos (NExT-QA val) plus the ST-Gap box clips, as the plan's fallback allows. Record this in the paper.


## 6. Videos: extract only what the project needs

The full NExT-QA video zip is large, so it is never copied into your Drive. The notebook reads it through the shortcut you added, extracts just the videos in our sets to the Colab disk, checks each one opens, and saves two compact zips to Drive:

- `data/videos_nextqa_eval.zip`: dev, main grid, descriptive and pilot videos
- `data/videos_intentqa_eval.zip`: IntentQA evaluation videos (Tier 2)

If no shortcut is found, the notebook tries a direct download with `gdown`. Public Google Drive files sometimes hit a download quota; if that happens, use the shortcut method.

In [8]:
VID_LOCAL = P['local_videos'] / 'nextqa'
VID_LOCAL.mkdir(parents=True, exist_ok=True)
GROUPS = {'videos_nextqa_eval.zip': ['dev', 'main_grid', 'descriptive', 'pilot'],
          'videos_intentqa_eval.zip': ['intentqa_eval']}

def find_video_source():
    shortcuts = sorted(p for p in P['source'].glob('*.zip') if 'annotation' not in p.name.lower())
    if shortcuts:
        log(f'using video source on Drive: {shortcuts[0].name}')
        return shortcuts[0]
    local = P['local_tmp'] / 'NExTVideo.zip'
    if not local.exists():
        log('no shortcut in data/source/; trying gdown (may hit the Google Drive quota)', 'WARN')
        pip_install('gdown')
        subprocess.run([sys.executable, '-m', 'gdown', '--id', CFG['sources']['nextqa_video_gdrive_id'],
                        '-O', str(local)], check=False)
    return local if local.exists() and zipfile.is_zipfile(local) else None

MISSING_PATH = ANN / 'videos_missing.json'      # videos absent from the source zip (known, not retried)

def zip_complete(zpath, vids):
    if not zpath.exists():
        return False
    with zipfile.ZipFile(zpath) as zf:
        names = {Path(n).stem for n in zf.namelist()}
    known_missing = set(load_json(MISSING_PATH, default=[]))
    return (set(vids) - known_missing) <= names

todo_groups = {z: sorted(set().union(*[V[n] for n in sets])) for z, sets in GROUPS.items()}
pending = {z: v for z, v in todo_groups.items() if not zip_complete(P['data'] / z, v)}

if not pending:
    log('both video zips on Drive are complete; nothing to extract')
else:
    src = find_video_source()
    if src is None:
        log('No video source available. Add the Drive shortcut (see the note above) and re-run this cell.', 'ERROR')
    else:
        with zipfile.ZipFile(src) as zf:
            index = {Path(n).stem: n for n in zf.namelist() if n.lower().endswith('.mp4')}
            wanted = sorted(set().union(*pending.values()))
            missing = [v for v in wanted if v not in index]

            def extract_one(vid):
                out = VID_LOCAL / f'{vid}.mp4'
                if out.exists():
                    return
                with zf.open(index[vid]) as s, open(out, 'wb') as d:
                    shutil.copyfileobj(s, d)

            res = run_resumable([v for v in wanted if v in index], key=str, fn=extract_one,
                                done={p.stem for p in VID_LOCAL.glob('*.mp4')}, desc='extract video',
                                progress_every=100)
            print(res)
            save_json(sorted(missing), MISSING_PATH)
            if missing:
                where = {n: sorted(set(missing) & V[n]) for n in V if set(missing) & V[n]}
                log(f'{len(missing)} needed videos are not in the source zip: {where}', 'WARN')
                if any(n in where for n in ['dev', 'main_grid', 'descriptive']):
                    log('Missing videos are in a core set: their questions will be scored as unanswerable. '
                        'Note them in the paper or replace them with FORCE_RESAMPLE before any results exist.', 'WARN')

        # probe every extracted video, then pack and copy the zips to Drive
        meta_path = ANN / 'video_meta.csv'
        meta = pd.read_csv(meta_path, dtype={'video_id': str}) if meta_path.exists() else pd.DataFrame(columns=['video_id'])
        known = set(meta.video_id)
        new_rows = [{'video_id': p.stem, **probe_video(p)} for p in sorted(VID_LOCAL.glob('*.mp4')) if p.stem not in known]
        if new_rows:
            meta = pd.concat([meta, pd.DataFrame(new_rows)], ignore_index=True)
            meta.to_csv(meta_path, index=False)
        bad = set(meta[meta.ok.astype(str) != 'True'].video_id)
        if bad:
            log(f'{len(bad)} videos failed to open: {sorted(bad)[:10]}', 'WARN')

        for zname, vids in pending.items():
            have = [v for v in vids if (VID_LOCAL / f'{v}.mp4').exists() and v not in bad]
            tmp = P['local_tmp'] / zname
            with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_STORED) as out:   # mp4 is already compressed
                for v in have:
                    out.write(VID_LOCAL / f'{v}.mp4', f'nextqa/{v}.mp4')
            shutil.copy2(tmp, P['data'] / zname)
            log(f'{zname}: {len(have)}/{len(vids)} videos, {_gb((P["data"] / zname).stat().st_size)} GB saved to Drive')

drive_gb = sum(f.stat().st_size for f in BASE.rglob('*') if f.is_file() and not f.is_symlink()) / 1024 ** 3
print(f'Project folder size on Drive: {drive_gb:.2f} GB (free Drive tier: 15 GB shared with Gmail and Photos)')

[09:51:43] INFO  using video source on Drive: NExTVideo.zip
[09:51:45] INFO  extract video: 634 total, 0 already done, 634 to do
[09:55:37] INFO  extract video: 100/634 | 2.3s each | ETA 20.6 min
[09:57:03] INFO  extract video: 200/634 | 1.6s each | ETA 11.5 min
[09:57:38] INFO  extract video: 300/634 | 1.2s each | ETA 6.6 min
[09:58:02] INFO  extract video: 400/634 | 0.9s each | ETA 3.7 min
[09:58:22] INFO  extract video: 500/634 | 0.8s each | ETA 1.8 min
[09:58:36] INFO  extract video: 600/634 | 0.7s each | ETA 0.4 min
{'total': 634, 'skipped': 0, 'ok': 634, 'failed': 0, 'remaining': 0, 'seconds': 413.3}
[09:59:13] INFO  videos_nextqa_eval.zip: 410/410 videos, 1.58 GB saved to Drive
[09:59:32] INFO  videos_intentqa_eval.zip: 224/224 videos, 1.02 GB saved to Drive
Project folder size on Drive: 25.35 GB (free Drive tier: 15 GB shared with Gmail and Photos)


## 7. ST-Gap: templates and clip registration

ST-Gap is recorded in weeks 2–4. This step creates the files the team fills in, and registers whatever clips have been uploaded so far. It is safe to re-run after each recording session.

**Naming rule:** `S01_short.mp4`, `S01_medium.mp4`, `S01_long.mp4` … `S20_long.mp4`, uploaded to `data/stgap/raw/`.

| File | Filled by | What goes in it |
| --- | --- | --- |
| `scenarios.csv` | script writer | one row per scenario: actors, props, the two key events and the distractor events |
| `event_log.csv` | annotator (CVAT) | one row per event per clip, with start/end times |
| `questions.csv` | a member who did **not** write the script | six questions per clip with five options |

In [9]:
SG = P['stgap']
(SG / 'raw').mkdir(parents=True, exist_ok=True)
TEMPLATES = {
    'scenarios.csv': ['scenario_id', 'title', 'setting', 'actors', 'props', 'key_event_a', 'key_event_b',
                      'distractors_medium', 'distractors_long', 'script_writer', 'notes'],
    'event_log.csv': ['clip_id', 'scenario_id', 'gap_version', 'event_id', 't_start_s', 't_end_s', 'actor',
                      'verb', 'object', 'location', 'is_key', 'is_distractor', 'annotator'],
    'questions.csv': ['clip_id', 'qnum', 'template', 'question', 'a0', 'a1', 'a2', 'a3', 'a4', 'answer_idx',
                      'evidence_event_ids', 'writer', 'leak_check_correct_orders'],
}
for name, cols in TEMPLATES.items():
    path = SG / name
    if not path.exists():
        pd.DataFrame(columns=cols).to_csv(path, index=False)
        log(f'created template {name}')
if not (SG / 'scenarios.csv').exists() or pd.read_csv(SG / 'scenarios.csv').empty:
    example = pd.DataFrame([{'scenario_id': 'S01', 'title': 'phone taken while owner is away', 'setting': 'living room',
                             'actors': 'P1; P2', 'props': 'phone; table', 'key_event_a': 'P1 puts phone on table',
                             'key_event_b': 'P2 takes phone', 'distractors_medium': 'P2 opens window',
                             'distractors_long': 'P2 opens window; P2 sits; P2 drinks water', 'script_writer': '',
                             'notes': 'EXAMPLE ROW: replace with real scenarios'}])
    example.to_csv(SG / 'scenarios.csv', index=False)

CLIP_RE = re.compile(r'^S(\d{2})_(short|medium|long)\.mp4$')
clips = []
for p in sorted((SG / 'raw').glob('*')):
    m = CLIP_RE.match(p.name)
    info = probe_video(p) if m else {}
    problems = []
    if not m:
        problems.append('name does not match S##_short|medium|long.mp4')
    else:
        if not info.get('ok'):
            problems.append('cannot be read')
        d = info.get('duration_s') or 0
        if not 30 <= d <= 90:
            problems.append(f'duration {d}s outside 30-90s')
        if min(info.get('width', 0), info.get('height', 0)) < 720:
            problems.append('below 720p')
    clips.append({'file': p.name, 'scenario_id': f'S{m.group(1)}' if m else '', 'gap_version': m.group(2) if m else '',
                  **info, 'problems': '; '.join(problems)})
C = pd.DataFrame(clips, columns=['file', 'scenario_id', 'gap_version', 'frames', 'fps', 'width', 'height',
                                 'duration_s', 'ok', 'problems'])
C.to_csv(SG / 'clip_registry.csv', index=False)

good = C[C.problems == '']
complete = good.groupby('scenario_id').gap_version.nunique()
print(f'ST-Gap clips uploaded: {len(C)} | valid: {len(good)} | complete triplets: {(complete == 3).sum()} of 20')
if (C.problems != '').any():
    print(C[C.problems != ''][['file', 'problems']].to_string(index=False))
if len(good):
    z = P['data'] / 'videos_stgap.zip'
    with zipfile.ZipFile(P['local_tmp'] / z.name, 'w', zipfile.ZIP_STORED) as out:
        for f in good.file:
            out.write(SG / 'raw' / f, f'stgap/{f}')
    shutil.copy2(P['local_tmp'] / z.name, z)
    log(f'videos_stgap.zip updated with {len(good)} clips')

[09:59:37] INFO  created template scenarios.csv
[09:59:37] INFO  created template event_log.csv
[09:59:37] INFO  created template questions.csv
ST-Gap clips uploaded: 0 | valid: 0 | complete triplets: 0 of 20


## 8. Data card and run record

In [10]:
card = {
    'created': dt.datetime.now().isoformat(timespec='seconds'),
    'sources': load_json(ANN / 'SOURCES.json'),
    'nextqa_questions': Q.groupby('split').size().to_dict(),
    'ambiguous_excluded': int(Q.ambiguous.sum()),
    'sets': {n: {'questions': len(d), 'videos': int(d.video_id.nunique()),
                 'types': d.type.value_counts().sort_index().to_dict()} for n, d in SETS.items()},
    'set_hashes': load_json(SUB / 'set_hashes.json'),
    'anchored_temporal_in_grid': int(len(anchored)),
    'vidor_coverage': load_json(SUB / 'vidor_coverage.json'),
    'video_zips': {z: (P['data'] / z).exists() for z in GROUPS},
    'stgap_valid_clips': int(len(good)),
    'intentqa_note': 'IntentQA test videos all come from NExT-QA train/val: QLoRA models are never evaluated on IntentQA.',
}
save_json(card, SUB / 'data_card.json')
RUN.finish('done', notes=f"sets={ {n: len(d) for n, d in SETS.items()} }")
print(json.dumps(card['sets'], indent=1))

[09:59:40] INFO  run finished: 01_data_prep-20261015-095028-f05c0b [done] 9.2 min, peak RAM 0.55 GB, peak VRAM 0.0 GB
{
 "dev": {
  "questions": 300,
  "videos": 92,
  "types": {
   "CH": 50,
   "CW": 110,
   "TC": 50,
   "TN": 70,
   "TP": 20
  }
 },
 "main_grid": {
  "questions": 600,
  "videos": 189,
  "types": {
   "CH": 100,
   "CW": 220,
   "TC": 100,
   "TN": 140,
   "TP": 40
  }
 },
 "descriptive": {
  "questions": 200,
  "videos": 129,
  "types": {
   "DC": 50,
   "DL": 70,
   "DO": 80
  }
 },
 "pilot": {
  "questions": 50,
  "videos": 10,
  "types": {
   "CH": 6,
   "CW": 16,
   "TC": 12,
   "TN": 14,
   "TP": 2
  }
 },
 "robustness": {
  "questions": 200,
  "videos": 126,
  "types": {
   "CH": 33,
   "CW": 74,
   "TC": 33,
   "TN": 47,
   "TP": 13
  }
 },
 "intentqa_eval": {
  "questions": 300,
  "videos": 224,
  "types": {
   "CH": 50,
   "CW": 176,
   "TN": 71,
   "TP": 3
  }
 }
}


## Week-1 checklist

- [ ] Section 3 printed the six sets with no shortfalls (or the shortfalls are recorded in `sampling_record.json`).
- [ ] `videos_nextqa_eval.zip` and `videos_intentqa_eval.zip` are on Drive, and no needed video is missing.
- [ ] The VidOR coverage numbers have been noted; if main-grid coverage is low, the E1 fallback applies.
- [ ] The anchor sheets are shared with the two annotators; marking can run alongside weeks 2–4.
- [ ] `scenarios.csv` has its 20 scenarios before ST-Gap recording starts in week 2.